# CSN Cancer Data Consolidation Pipeline
This notebook reads all raw/translated scrapings from Dutch cancer resources, applies custom linguistic clean-ups, repairs character encoding corruptions (Mojibake), standardizes the categories, and dynamically maps them to the canonical **CSN Patient Question Bank v3** definitions using custom rules and text similarity metrics.

In [30]:
import pandas as pd
import numpy as np
import re

print("Libraries successfully imported!")

Libraries successfully imported!


## 1. Character Encoding Repair (Mojibake Remediation)
Define a cleaning matrix to repair corruptions occurring when `UTF-8` data is compressed or improperly loaded into Windows-1252 layouts.

In [31]:
def fix_mojibake(text):
    if not isinstance(text, str):
        return text
    
    replacements = {
        '‚Ä¢': '•',
        '¬Æ': '®',
        '¬†': ' ',
        '¬±': '±',
        '¬™': '™',
        '¬©': '©',
        'â€¢': '•',
        'â€ô': "'",
        'â€œ': '"',
        'â€\x9d': '"',
        'â€“': '–',
        'â€”': '—'
    }
    for old, new in replacements.items():
        text = text.replace(old, new)
    return text.replace('\xad', '')

def clean_question_prefix(text):
    if not isinstance(text, str):
        return text
    
    # Strip awkward administrative layout wraps
    prefix = "Is treatment/reimbursement available for Advice - whether or not to reimburse "
    if text.startswith(prefix):
        cleaned = text[len(prefix):]
        if cleaned.endswith("?"):
            cleaned = cleaned[:-1]
        return f"Is {cleaned} reimbursed by Dutch insurance?"
        
    # Clean Zorginstituut evaluation patterns
    bad_prefix1 = "Is treatment/reimbursement available for Evaluation of the "
    bad_prefix2 = "Is treatment/reimbursement available for Evaluation position "
    bad_prefix3 = "Is treatment/reimbursement available for Evaluation of "
    
    if text.startswith(bad_prefix1):
        return f"What is the reimbursement status for the {text[len(bad_prefix1):].replace('?', '')}?"
    elif text.startswith(bad_prefix2):
        return f"What is the reimbursement status for {text[len(bad_prefix2):].replace('?', '')}?"
    elif text.startswith(bad_prefix3):
        return f"What is the reimbursement status for {text[len(bad_prefix3):].replace('?', '')}?"
        
    return text

## 2. Dutch to English Medical Classifications Translation
Explicitly translate all remaining native Dutch tumor and care entities into standard English clinical terminology.

In [32]:
type_mapping = {
    'Algemeen / Onbekend': 'All / General',
    'Borstkanker (Breast)': 'Breast cancer',
    'Longkanker (Lung)': 'Lung cancer',
    'Lymfeklierkanker (Lymphoma)': 'Lymphoma',
    'Bloedkanker (Leukemia)': 'Leukemia',
    'Prostaatkanker (Prostate)': 'Prostate cancer',
    'Darmkanker (Colorectal)': 'Colorectal cancer',
    'Hoofd-halskanker (Head & Neck)': 'Head & Neck cancer',
    'Alvleesklier-kanker': 'Pancreatic cancer',
    'Baarmoederhalskanker': 'Cervical cancer',
    'Detecting colon cancer': 'Colon cancer',
    'Borstkanker bij mannen': 'Breast cancer (male)',
    'Borstkanker': 'Breast cancer',
    'Alvleesklierkanker': 'Pancreatic cancer',
    'Baarmoederkanker': 'Uterine cancer',
    'Baarmoedersarcoom': 'Uterine sarcoma',
    'Basaalcelcarcinoom': 'Basal cell carcinoma',
    'Bijnierkanker': 'Adrenal cancer',
    'Buikvlieskanker': 'Peritoneal cancer',
    'Botkanker (botsarcoom)': 'Bone cancer (sarcoma)',
    'Blaaskanker': 'Bladder cancer',
    'Darmkanker (dikkedarmkanker)': 'Colorectal cancer',
    'Darmkanker': 'Colorectal cancer',
    'Dunnedarmkanker': 'Small intestine cancer',
    'Galblaaskanker': 'Gallbladder cancer',
    'Hartkanker': 'Heart cancer',
    'Endeldarmkanker': 'Rectal cancer',
    'Eierstokkanker': 'Ovarian cancer',
    'Galwegkanker': 'Bile duct cancer',
    'Hoofd-halskanker': 'Head & Neck cancer',
    'Huidkanker': 'Skin cancer',
    'Gastro-intestinale stromale tumor': 'GIST (Gastrointestinal stromal tumor)',
    'Kanker bij kinderen': 'Childhood cancer',
    'Hodgkinlymfoom': 'Hodgkin lymphoma',
    'Hoge urinewegkanker': 'Upper urinary tract cancer',
    'Hersentumoren': 'Brain tumors',
    'Leukemie': 'Leukemia',
    'Keelkanker': 'Throat cancer',
    'Huidlymfoom': 'Skin lymphoma',
    'Lymfeklierkanker': 'Lymphoma',
    'Lipkanker': 'Lip cancer',
    'Leverkanker': 'Liver cancer',
    'Luchtpijpkanker': 'Trachea cancer',
    'Longkanker': 'Lung cancer',
    'Maagkanker': 'Stomach cancer',
    'Merkelcelcarcinoom': 'Merkel cell carcinoma',
    'MPN (myeloproliferatieve neoplasmata)': 'MPN (Myeloproliferative neoplasms)',
    'Melanoom': 'Melanoma',
    'Mesothelioom': 'Mesothelioma',
    'Mondkanker': 'Oral cancer',
    'Neuro-endocriene tumoren': 'Neuroendocrine tumors',
    'Neuro-endocriene neoplasie': 'Neuroendocrine neoplasia',
    'Multipel myeloom (ziekte van Kahler)': 'Multiple myeloma',
    'Myelodysplastisch syndroom': 'Myelodysplastic syndrome (MDS)',
    'Mucosaal melanoom': 'Mucosal melanoma',
    'Non-hodgkinlymfoom': 'Non-Hostgkin lymphoma',
    'Neuskanker': 'Nasal cancer',
    'Papil van Vater-kanker': 'Ampullary cancer (Ampulla of Vater)',
    'Nierkanker': 'Kidney cancer',
    'Oogmelanoom': 'Ocular melanoma',
    'Oogkanker': 'Eye cancer',
    'Oorkanker': 'Ear cancer',
    'Plaveiselcelcarcinoom': 'Squamous cell carcinoma',
    'Primaire tumor onbekend (PTO)': 'Unknown primary tumor',
    'Primaire Tumor Onbekend (PTO)': 'Unknown primary tumor',
    'Peniskanker': 'Penile cancer',
    'Plasbuiskanker': 'Urethral cancer',
    'Pseudomyxoma peritonei (PMP)': 'Pseudomyxoma peritonei (PMP)',
    'Prostaatkanker': 'Prostate cancer',
    'Schaamlipkanker (vulvakanker)': 'Vulvar cancer',
    'Schildklierkanker': 'Thyroid cancer',
    'Slokdarmkanker': 'Esophageal cancer',
    'Slokdarm- en maagkanker': 'Esophageal & Stomach cancer',
    'Speekselklierkanker': 'Salivary gland cancer',
    'Thymuskanker': 'Thymus cancer',
    'Talgkliercarcinoom': 'Sebaceous gland carcinoma',
    'Trofoblastziekten': 'Trophoblastic disease',
    'Strottenhoofdkanker': 'Laryngeal cancer',
    'Tongkanker': 'Tongue cancer',
    'Zeldzame kankers': 'Rare cancers',
    'Zeldzame kanker': 'Rare cancer',
    'Urachuskanker': 'Urachal cancer',
    'Ziekte van Waldenström': 'Waldenstrom macroglobulinemia',
    'Zaadbalkanker (teelbalkanker)': 'Testicular cancer',
    'Vaginakanker': 'Vaginal cancer',
    'Wekedelentumoren / wekedelensarcomen': 'Soft tissue sarcoma',
    'Bot- en wekedelenkanker': 'Bone and soft tissue cancer',
    'Atypisch fibroxanthoom': 'Atypical fibroxanthoma',
    'Anuskanker': 'Anal cancer',
    'Hemato-oncologie': 'Hemato-oncology',
    'HPB-tumoren': 'HPB (Hepato-pancreato-biliary) tumors',
    'Kanker bij jongvolwassenen': 'Young adult cancer'
}

## 3. Consolidation and Unpivoting Stage
Compile data from all seven raw files, unpivot wide multi-field rows into discrete data entities, and construct the basic dataset.

In [33]:
all_rows = []

# --- Standard template schema setup to prevent ValueError ---
target_cols = [
    'row_id', 'source_name', 'source_url', 'trustworthiness', 'cancer_type',
    'topic', 'journey_stage', 'patient_question', 'question_id',
    'english_summary', 'english_full', 'original_dutch', 'content_hash',
    'scraped_at', 'is_latest', 'translator', 'validated_by', 'notes'
]

# 1. Zorgwijzer
try:
    df = pd.read_csv('zorgwijzer_cancer_data_translated.csv')
    for idx, row in df.iterrows():
        all_rows.append({
            'source_name': 'Zorgwijzer',
            'source_url': row.get('original_url'),
            'trustworthiness': row.get('trustworthiness'),
            'cancer_type': row.get('source_organisation'),
            'topic': row.get('care_pathway_topic_en'),
            'journey_stage': row.get('patient_journey_stage_en'),
            'patient_question': row.get('source_page_title_nl_en'),
            'english_summary': row.get('action_or_next_step_en'),
            'english_full': row.get('original_dutch_text_en'),
            'original_dutch': row.get('original_dutch_text'),
            'content_hash': row.get('content_hash'),
            'scraped_at': row.get('scrape_date'),
            'is_latest': True,
            'translator': 'Machine',
            'validated_by': None,
            'notes': f"Stakeholder: {row.get('stakeholder_en')}"
        })
except Exception as e:
    print("Zorgwijzer missing or failed:", e)

# 2. Zorginstituut
try:
    df = pd.read_csv('zorginstituut_categorized_cancer_data_translated.csv')
    for idx, row in df.iterrows():
        all_rows.append({
            'source_name': 'Zorginstituut Nederland',
            'source_url': row.get('Source URL'),
            'trustworthiness': row.get('trustworthiness'),
            'cancer_type': row.get('Standardized Cancer Type'),
            'topic': row.get('Advisory Topic_en'),
            'journey_stage': 'Treatment',
            'patient_question': f"Is treatment/reimbursement available for Advice - whether or not to reimburse {row.get('Advisory Topic_en')}?",
            'english_summary': row.get('Intro Summary_en'),
            'english_full': row.get('Full Context Block_en'),
            'original_dutch': row.get('Full Context Block'),
            'content_hash': row.get('Content Hash'),
            'scraped_at': row.get('Scrape Date'),
            'is_latest': True,
            'translator': 'Machine',
            'validated_by': None,
            'notes': None
        })
except Exception as e:
    print("Zorginstituut missing or failed:", e)

# 3. Thuisarts
try:
    df = pd.read_csv('thuisarts_comprehensive_cancer_data_translated.csv')
    for idx, row in df.iterrows():
        all_rows.append({
            'source_name': 'Thuisarts',
            'source_url': row.get('URL'),
            'trustworthiness': row.get('trustworthiness'),
            'cancer_type': row.get('Topic Category_en'),
            'topic': row.get('Page Title_en'),
            'journey_stage': None,
            'patient_question': row.get('Page Title_en'),
            'english_summary': row.get('In Brief Summary_en'),
            'english_full': row.get('Content Extract_en'),
            'original_dutch': row.get('Content Extract'),
            'content_hash': row.get('Content Hash'),
            'scraped_at': row.get('Scrape Date'),
            'is_latest': True,
            'translator': 'Machine',
            'validated_by': None,
            'notes': None
        })
except Exception as e:
    print("Thuisarts missing or failed:", e)

# 4. NFK
try:
    df = pd.read_csv('nfk_nl_table_translated.csv')
    for idx, row in df.iterrows():
        all_rows.append({
            'source_name': 'NFK',
            'source_url': row.get('Source_URL'),
            'trustworthiness': row.get('trustworthiness'),
            'cancer_type': row.get('Cancer_type'),
            'topic': row.get('Topic_en'),
            'journey_stage': row.get('Journey_stage'),
            'patient_question': row.get('Question_en'),
            'english_summary': row.get('Next_step_en'),
            'english_full': row.get('General_description_en'),
            'original_dutch': row.get('General_description'),
            'content_hash': row.get('Content_hash'),
            'scraped_at': row.get('Date'),
            'is_latest': True,
            'translator': 'Machine',
            'validated_by': None,
            'notes': f"Keywords: {row.get('Keywords')}"
        })
except Exception as e:
    print("NFK missing or failed:", e)

# 5. Kanker.nl Breakout Matrix (Unpivot Matrix)
try:
    df = pd.read_csv('kanker_nl_master_table_breakout_v3_translated.csv')
    breakout_fields = [
        ('General Description', 'General Description', None),
        ('Prognosis (Vooruitzichten)', 'Prognosis', None),
        ('Symptoms (Symptomen)', 'Symptoms', 'Pre-Dx'),
        ('Causes (Oorzaken)', 'Causes', None),
        ('Treatments (Behandeling)', 'Treatment', 'Tx')
    ]
    for idx, row in df.iterrows():
        for field_nl, topic_name, stage in breakout_fields:
            field_en = f"{field_nl}_en"
            if pd.notna(row.get(field_en)) and str(row.get(field_en)).strip() != '':
                all_rows.append({
                    'source_name': 'Kanker.nl',
                    'source_url': row.get('Source URL'),
                    'trustworthiness': row.get('trustworthiness'),
                    'cancer_type': row.get('Cancer Type'),
                    'topic': topic_name,
                    'journey_stage': stage,
                    'patient_question': f"What are the {topic_name.lower()} of/for {row.get('Cancer Type')}?",
                    'english_summary': None,
                    'english_full': row.get(field_en),
                    'original_dutch': row.get(field_nl),
                    'content_hash': row.get('Content Hash'),
                    'scraped_at': row.get('Scrape Date'),
                    'is_latest': True,
                    'translator': 'Machine',
                    'validated_by': None,
                    'notes': f"Original field: {field_nl}"
                })
except Exception as e:
    print("Kanker.nl breakout missing or failed:", e)

# 6. IKNL Matrix (Unpivot Matrix)
try:
    df = pd.read_csv('iknl_master_table_translated.csv')
    iknl_fields = [
        ('General Description', 'General Description', None),
        ('Decision-making', 'Decision-making', None),
        ('Treatment', 'Treatment', 'Tx'),
        ('Statistics & Survival', 'Statistics & Survival', None),
        ('Life after cancer', 'Life after cancer', 'Support'),
        ('Palliative phase', 'Palliative phase', None),
        ('Research', 'Research', None)
    ]
    for idx, row in df.iterrows():
        for field_nl, topic_name, stage in iknl_fields:
            field_en = f"{field_nl}_en"
            if pd.notna(row.get(field_en)) and str(row.get(field_en)).strip() != '':
                all_rows.append({
                    'source_name': 'IKNL',
                    'source_url': row.get('Source URL'),
                    'trustworthiness': row.get('trustworthiness'),
                    'cancer_type': row.get('Cancer Type'),
                    'topic': topic_name,
                    'journey_stage': stage,
                    'patient_question': f"Information regarding {topic_name.lower()} for {row.get('Cancer Type')}",
                    'english_summary': None,
                    'english_full': row.get(field_en),
                    'original_dutch': row.get(field_nl),
                    'content_hash': row.get('Content Hash'),
                    'scraped_at': row.get('Scrape Date'),
                    'is_latest': True,
                    'translator': 'Machine',
                    'validated_by': None,
                    'notes': f"Original field: {field_nl}"
                })
except Exception as e:
    print("IKNL missing or failed:", e)

# 7. Additional Narrative Data
try:
    df = pd.read_csv('additional_cancer_information_translated.csv')
    for idx, row in df.iterrows():
        all_rows.append({
            'source_name': 'Kanker.nl (Additional)',
            'source_url': row.get('Source URL'),
            'trustworthiness': row.get('trustworthiness'),
            'cancer_type': 'All / General',
            'topic': row.get('Global Track'),
            'journey_stage': None,
            'patient_question': row.get('Page Title_en'),
            'english_summary': None,
            'english_full': row.get('Article Content_en'),
            'original_dutch': row.get('Article Content'),
            'content_hash': row.get('Content Hash'),
            'scraped_at': row.get('Scrape Date'),
            'is_latest': True,
            'translator': 'Machine',
            'validated_by': None,
            'notes': None
        })
except Exception as e:
    print("Additional information failed:", e)

# --- CRITICAL FIX: Initialize with template column framework ---
master_df = pd.DataFrame(all_rows, columns=target_cols)
print(f"Initial combined row count: {len(master_df)}")

Initial combined row count: 857


## 4. Semantic Parsing and CSN Question Bank Linkage
Load the approved `CSN_question_bank_v3` reference table and map row items to valid canonical question keys using regex strings and keyword parameters.

In [34]:

# --- Load Question Bank Reference Tables ---
questions_list = []
loaded_files = []

# 1. Try loading the primary file
try:
    q_bank_primary = pd.read_csv('question_bank.csv')
    questions_list.extend(q_bank_primary.to_dict('records'))
    loaded_files.append("question_bank.csv")
except Exception as e:
    print("Notice: 'question_bank.csv' not found or could not be read.")

# 2. Try loading the alternative/secondary file
try:
    q_bank_alt = pd.read_csv('CSN_question_bank_v3.xlsx - Questions.csv')
    questions_list.extend(q_bank_alt.to_dict('records'))
    loaded_files.append("CSN_question_bank_v3.xlsx - Questions.csv")
except Exception as e:
    print("Notice: 'CSN_question_bank_v3.xlsx - Questions.csv' not found or could not be read.")

# Deduplicate questions based on question_id if files overlapped
if questions_list:
    # Using a dict comprehension to keep the last seen instance of each question_id
    questions_list = list({q.get('question_id'): q for q in questions_list if q.get('question_id')}.values())
    print(f"Successfully loaded and unified {len(questions_list)} unique canonical questions from: {', '.join(loaded_files)}")
else:
    print("Warning: No Question Bank files found, running in structural fallback mode.")


# --- Matching Logic ---
def normalize_text(text):
    if not isinstance(text, str):
        return ""
    return re.sub(r'[^\w\s]', '', text.lower()).strip()

def map_to_question_bank(row):
    if len(questions_list) == 0:
        return None
    q_text = normalize_text(row.get('patient_question', ''))
    full_text = normalize_text(row.get('english_full', ''))
    
    best_match_id = None
    best_score = 0
    
    for qb in questions_list:
        qb_text = normalize_text(qb.get('question_text', ''))
        overlap = len(set(qb_text.split()).intersection(set(q_text.split())))
        score = overlap
        
        # Custom matching boost layers matching intended scope patterns
        if 'reimburse' in q_text or 'insurance' in q_text:
            if qb.get('question_id') in ['Q-046', 'Q-047', 'Q-048']:
                score += 5
        if 'palliative' in q_text or 'incurable' in q_text or 'palliative' in full_text:
            if qb.get('question_id') == 'Q-067':
                score += 8
        if 'work' in q_text or 'working' in q_text:
            if qb.get('question_id') in ['Q-051', 'Q-052']:
                score += 8
        if 'second opinion' in q_text or 'second opinion' in full_text:
            if qb.get('topic') == 'Second Opinions':
                score += 10
        if 'waiting' in q_text or 'how long' in q_text:
            if qb.get('topic') == 'Waiting Times':
                score += 5
                
        if score > best_score and score >= 2:
            best_score = score
            best_match_id = qb.get('question_id')
            
    return best_match_id

# --- Apply Mapping ---
# Bulletproof protection check against ValueError for empty DataFrames
if not master_df.empty:
    master_df['question_id'] = master_df.apply(map_to_question_bank, axis=1)
else:
    master_df['question_id'] = pd.Series(dtype='object')

print("Mapping vectors complete!")

Notice: 'CSN_question_bank_v3.xlsx - Questions.csv' not found or could not be read.
Successfully loaded and unified 67 unique canonical questions from: question_bank.csv
Mapping vectors complete!


## 5. Strict Structural Formatting and Filtering
Apply fallback criteria to organize the datasets under the strict canonical schemas. Clean verbose string fragments and compile everything into a database table structure.

In [35]:
def apply_strict_canonical_rules(row):
    # Standardized parameters from question bank guidelines
    valid_topics = [
        'Referrals and process', 'Waiting Times', 'Treatment Decisions', 
        'Second Opinions', 'Additional Care & Support', 'Communication & Understanding'
    ]
    valid_stages = ['Pre-Diagnosis', 'Diagnosis', 'Treatment', 'Post-Treatment', 'Support', 'First 48 Hours']
    
    # Pre-empt advanced/incurable themes or prognostic figures
    full_text_en = str(row.get('english_full', '')).lower() if pd.notna(row.get('english_full')) else ""
    q_text_en = str(row.get('patient_question', '')).lower() if pd.notna(row.get('patient_question')) else ""
    
    incurable_keywords = ['incurable', 'cannot get better', 'cannot be cured', 'metastatic', 'palliative', 'final phase of life']
    is_incurable = any(k in full_text_en or k in q_text_en for k in incurable_keywords)
    
    stats_keywords = ['prognosis', 'outlook', 'survival', 'statistics', 'figures']
    is_stats = any(k in full_text_en or k in q_text_en for k in stats_keywords)
    
    # Explicit topic fallback override
    t_str = str(row.get('topic', '')).strip()
    if t_str.startswith("Advice -") or t_str.startswith("Evaluation") or "reimburse" in t_str.lower():
        row['topic'] = 'Advice'
        
    if is_incurable:
        row['topic'] = 'Advanced/Incurable Care'
        row['journey_stage'] = 'Support'
    elif is_stats:
        row['topic'] = 'Statistics & Survival'
        if pd.isna(row.get('journey_stage')):
            row['journey_stage'] = 'Diagnosis'
            
    # --- UPDATED STEP 1: Search the unified questions_list and pull the text ---
    current_q_id = row.get('question_id')
    row['canonical_question'] = None  # Initialize the column
    
    if pd.notna(current_q_id) and 'questions_list' in globals() and questions_list:
        matched = next((q for q in questions_list if q.get('question_id') == current_q_id), None)
        if matched:
            row['topic'] = matched.get('topic', row.get('topic'))
            row['journey_stage'] = str(matched.get('journey_stage', row.get('journey_stage'))).strip()
            # Extract and assign the actual question text here
            row['canonical_question'] = matched.get('question_text', '')  
    else:
        # Map unlinked categories cleanly
        t = str(row.get('topic', ''))
        if t in ['Advanced/Incurable Care', 'Life after cancer', 'Fatigue & Consequences']:
            row['topic'] = 'Additional Care & Support'
        elif t in ['Advice', 'Treatment', 'Treatments & Radiotherapy']:
            row['topic'] = 'Treatment Decisions'
        elif t in ['General Description', 'Symptoms', 'Causes', 'Statistics & Survival']:
            row['topic'] = 'Communication & Understanding'
            
    # Fallback assignment logic: if journey stage is empty, inherit the current topic parameter
    if pd.isna(row.get('journey_stage')) or str(row.get('journey_stage')).strip().lower() in ['', 'nan']:
        row['journey_stage'] = row.get('topic', 'Support')
        
    # Step 2: Enforce strict categorical formatting boundaries
    t_curr = str(row.get('topic', '')).strip()
    s_curr = str(row.get('journey_stage', '')).strip()
    
    if t_curr not in valid_topics:
        if 'treat' in t_curr.lower() or 'decision' in t_curr.lower() or 'advice' in t_curr.lower():
            row['topic'] = 'Treatment Decisions'
        elif any(k in t_curr.lower() for k in ['research', 'support', 'care', 'fatigue', 'life']):
            row['topic'] = 'Additional Care & Support'
        else:
            row['topic'] = 'Communication & Understanding'
            
    if s_curr not in valid_stages:
        if 'pre' in s_curr.lower() or 'symptom' in s_curr.lower() or 'cause' in s_curr.lower():
            row['journey_stage'] = 'Pre-Diagnosis'
        elif 'diag' in s_curr.lower() or 'description' in s_curr.lower():
            row['journey_stage'] = 'Diagnosis'
        elif 'treat' in s_curr.lower() or 'tx' in s_curr.lower() or 'surgery' in s_curr.lower() or 'operation' in s_curr.lower():
            row['journey_stage'] = 'Treatment'
        elif any(k in s_curr.lower() for k in ['post', 'after', 'recover', 'life']):
            row['journey_stage'] = 'Post-Treatment'
        else:
            row['journey_stage'] = 'Support'
            
    return row

# Apply translations and text normalization fixes
if not master_df.empty:
    if 'cancer_type' in master_df.columns and 'type_mapping' in globals():
        master_df['cancer_type'] = master_df['cancer_type'].replace(type_mapping).fillna('All / General')

    for col in ['patient_question', 'english_full', 'english_summary', 'topic', 'journey_stage', 'original_dutch']:
        if col in master_df.columns and 'fix_mojibake' in globals():
            master_df[col] = master_df[col].apply(fix_mojibake)

    if 'patient_question' in master_df.columns and 'clean_question_prefix' in globals():
        master_df['patient_question'] = master_df['patient_question'].apply(clean_question_prefix)
    
    master_df = master_df.apply(apply_strict_canonical_rules, axis=1)

# --- UPDATED TARGET COLS: Added 'canonical_question' next to question_id ---
target_cols = [
    'row_id', 'source_name', 'source_url', 'trustworthiness', 'cancer_type',
    'topic', 'journey_stage', 'patient_question', 'question_id', 'canonical_question',
    'english_summary', 'english_full', 'original_dutch', 'content_hash',
    'scraped_at', 'is_latest', 'translator', 'validated_by', 'notes'
]

if 'row_id' in master_df.columns:
    master_df = master_df.drop(columns=['row_id'])
master_df.insert(0, 'row_id', range(1, len(master_df) + 1))

for col in target_cols:
    if col not in master_df.columns:
        master_df[col] = None
        
final_structured_df = master_df[target_cols]

# Write clean results out to database target csv file 
final_structured_df.to_csv('grouped_cancer_data_final_v2.csv', index=False)

print("=== PIPELINE OPERATION SUCCESSFUL ===")
print(f"Total Clean Structured Nodes Compiled: {len(final_structured_df)}")

=== PIPELINE OPERATION SUCCESSFUL ===
Total Clean Structured Nodes Compiled: 857


In [36]:
def transform_for_tableau(input_file, output_file):
    # Load the processed final v2 dataset
    df = pd.read_csv(input_file, low_memory=False)
    
    # --- UPDATED: Added 'canonical_question' here to clean its spaces and formatting ---
    text_columns = ['english_full', 'original_dutch', 'english_summary', 'notes', 'patient_question', 'canonical_question']
    
    for col in text_columns:
        if col in df.columns:
            # Step 1: Force column types to string and handle nulls gracefully
            df[col] = df[col].fillna("").astype(str)
            
            # Step 2: Swap breaking newlines (\n or \r) with safe web line-breaks (<br>)
            df[col] = df[col].str.replace(r'\r\n|\r|\n', ' <br> ', regex=True)
            
            # Step 3: Clean up multiple padded spaces caused by formatting shifts
            df[col] = df[col].str.replace(r'\s+', ' ', regex=True).str.strip()
            
    # Step 4: Export with aggressive double-quoting constraints to safeguard schemas
    df.to_csv(output_file, index=False, quoting=1, encoding='utf-8')
    print(f"Success! Cleaned file optimized for Tableau exported to: {output_file}")

# Execute the optimizer loop
transform_for_tableau('grouped_cancer_data_final_v2.csv', 'grouped_cancer_data_tableau_ready.csv')

Success! Cleaned file optimized for Tableau exported to: grouped_cancer_data_tableau_ready.csv
